# Responsible AI Resume Screening — End-to-End Notebook

**Author:** Nick Warshak  
**System:** Applicant shortlist ranking for software engineering requisitions  
**Classification:** High-risk AI system (EU AI Act Annex III §4; NYC Local Law 144 AEDT)

---

## The argument this notebook makes

Resume screening is the domain where AI hiring tools have most publicly failed. Amazon
scrapped an internal recruiting model in 2018 after finding it penalized resumes containing
the word "women's." The failure was not a bug. The model worked exactly as designed: it
learned to reproduce ten years of human hiring decisions, and those decisions were biased.

This notebook rebuilds that failure deliberately, measures it, and then fixes it — so that
each governance control can be evaluated against evidence rather than asserted.

**The central finding, stated up front:** the model that best predicts historical recruiter
behavior is *not* the model that best identifies qualified candidates. Optimizing the metric
available in production selects the discriminatory model.

## 1. Setup and data generation

Real resume corpora contain PII and cannot lawfully be repurposed for model training. We
generate a synthetic population in which *true qualification is known by construction* —
which is what makes the fairness audit verifiable. In real data, true qualification is never
observed, so you can never prove the model is wrong; here we can.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

from generate_data import generate, MERIT_FEATURES, PROXY_FEATURES, GOVERNED_FEATURES, NAIVE_FEATURES

df = generate(n=12000, seed=42)
print(f"{len(df):,} applicants, {df.advanced_to_onsite.mean():.1%} advanced to onsite")
df.head()

12,000 applicants, 24.8% advanced to onsite


,applicant_id,years_experience,education_level,num_relevant_skills,keyword_match_score,gpa,num_certifications,portfolio_projects,num_prior_roles,avg_tenure_months,...,top_school,referral,distance_from_office_km,sex,race_ethnicity,age_group,disability_disclosed,advanced_to_onsite,advanced_counterfactual,latent_qualification
0,A100000,3.2,1,5,0.254,2.76,1,0,3,15.1,...,0,1,2.5,Female,Asian,Under 40,0,0,0,-0.7788
1,A100001,7.8,1,9,0.592,3.23,1,1,2,52.2,...,0,1,4.2,Male,White,Under 40,0,1,0,0.5917
2,A100002,7.2,3,11,0.626,3.60,1,6,3,33.6,...,1,0,23.6,Female,White,Under 40,0,1,1,1.6115
3,A100003,5.6,2,12,0.738,3.47,1,5,4,14.6,...,0,0,21.9,Male,Two or more/Other,Under 40,0,1,1,1.4280
4,A100004,4.0,2,6,0.280,2.89,0,1,3,19.3,...,0,0,7.1,Male,White,Under 40,0,0,0,-0.5308


### The four bias channels

The label is **not** "was this person good at the job." It is "did a recruiter advance them."
That gap — a proxy label standing in for the construct of interest — is the single most
important governance flaw in commercial resume screeners. Four documented mechanisms are
injected:

| Channel | Mechanism | Real-world basis |
|---|---|---|
| Referral homophily | Referrals flow through existing employee networks | Rubineau & Fernandez (2013) |
| Employment-gap penalty | Gaps penalized; caregiving unequally distributed | BLS time-use data |
| Prestige proxy | "Top school" tracks parental income, which is racialized | Chetty et al. (2020) |
| Residual direct bias | Name-based callback discrimination | Bertrand & Mullainathan (2004) |

Critically, **latent qualification is generated independent of protected class.** In this
world there is no real ability difference between groups, so every disparity the audit finds
is pure measurement bias.

In [2]:
print("Advance rate by sex:")
print(df.groupby("sex").advanced_to_onsite.mean().round(4))
print("\nAdvance rate by race/ethnicity:")
print(df.groupby("race_ethnicity").advanced_to_onsite.mean().round(4))
print("\nMean TRUE qualification by race (all ~0 -> no real ability gap):")
print(df.groupby("race_ethnicity").latent_qualification.mean().round(4))

Advance rate by sex:
sex
Female    0.1758
Male      0.2759
Name: advanced_to_onsite, dtype: float64

Advance rate by race/ethnicity:
race_ethnicity
Asian                0.2677
Black                0.1571
Hispanic/Latino      0.1716
Two or more/Other    0.1507
White                0.2767
Name: advanced_to_onsite, dtype: float64

Mean TRUE qualification by race (all ~0 -> no real ability gap):
race_ethnicity
Asian               -0.0206
Black               -0.0319
Hispanic/Latino     -0.0353
Two or more/Other    0.0065
White               -0.0016
Name: latent_qualification, dtype: float64


## 2. Training: two models, two philosophies

**Naive** — every available feature, including known demographic proxies. The
"maximize AUC and ship it" model.

**Governed** — proxy features removed, calibrated, abstention band applied.

In [3]:
from train import run as train_run
metrics = train_run()

                                   MODEL PERFORMANCE                                    
model                 AUC(hist)      AP   Brier  AUC(true qual)  AUC(fair)
----------------------------------------------------------------------------------------
naive_logistic           0.9114  0.7746  0.0977          0.9151     0.8450
naive_gbm                0.9077  0.7654  0.1009          0.9215     0.8496
governed_logistic        0.8648  0.6795  0.1222          0.9611     0.8893
governed_gbm             0.8594  0.6630  0.1246          0.9522     0.8776
----------------------------------------------------------------------------------------
AUC(hist)      = ranks applicants the way past recruiters did
AUC(true qual) = ranks applicants by actual latent ability

Operating point @ 25% shortlist rate:
  naive_logistic       precision=0.578  recall=0.879  f1=0.698
  naive_gbm            precision=0.697  recall=0.716  f1=0.706
  governed_logistic    precision=0.556  recall=0.742  f1=0.635
  governe

### Reading the result

Compare the two AUC columns. The naive model wins on `AUC(hist)` — predicting what
recruiters did. The governed model wins on `AUC(true qual)` — identifying who was actually
qualified.

In production **you only ever observe the first column.** Standard model selection therefore
picks the naive model, and picks the worse one. This is the mechanism by which a
well-intentioned team ships a discriminatory system while following every ML best practice.

## 3. Fairness audit

Implements what NYC Local Law 144 requires of automated employment decision tools:
selection rates and impact ratios by sex, race/ethnicity, and their intersection.

An impact ratio below 0.80 is prima facie evidence of adverse impact under the EEOC Uniform
Guidelines (29 CFR 1607.4D).

In [4]:
from fairness import run as fairness_run
audit = fairness_run()

                   DISPARATE IMPACT AUDIT  (four-fifths threshold = 0.80)                   

### BASELINE: historical human recruiter decisions
 group    n  selection_rate  impact_ratio  passes_4_5ths
  Male 1732          0.2789        1.0000           True
Female  668          0.1692        0.6066          False
            group    n  selection_rate  impact_ratio  passes_4_5ths
            White 1113          0.2830        1.0000           True
            Asian  739          0.2530        0.8941           True
  Hispanic/Latino  262          0.1756        0.6204          False
Two or more/Other   92          0.1739        0.6145          False
            Black  194          0.1649        0.5828          False

### NAIVE MODEL (all features incl. proxies)   [naive_gbm]

-- Sex --
 group    n  selection_rate  impact_ratio    tpr    fpr  passes_4_5ths
  Male 1732          0.2864        1.0000 0.5831 0.1133           True
Female  668          0.1751        0.6116 0.3891 0.0559        

### Statistical honesty

An impact ratio computed on 68 applicants is not the same evidence as one computed on 1,113.
Reporting a bare point estimate as a compliance finding overstates what the audit knows.

In [5]:
from significance import run as sig_run
sig_run()

               BOOTSTRAP CONFIDENCE INTERVALS ON IMPACT RATIOS  (5,000 resamples)               
model               attribute         group                            n   ratio            95% CI  P(<0.80)
------------------------------------------------------------------------------------------------


governed_logistic   sex               Female                         668   0.956    [0.849, 1.000]     0.003


governed_logistic   race_ethnicity    Black                          194   0.883    [0.660, 1.000]     0.223


governed_logistic   race_ethnicity    Hispanic/Latino                262   0.811    [0.612, 1.000]     0.463


governed_logistic   intersection      Female / Hispanic/Latino        68   0.587    [0.343, 0.887]     0.924


governed_logistic   intersection      Female / Black                  60   0.853    [0.547, 1.000]     0.337


naive_gbm           sex               Female                         668   0.614    [0.509, 0.728]     0.999


naive_gbm           race_ethnicity    Black                          194   0.702    [0.501, 0.917]     0.826


naive_gbm           intersection      Female / Black                  60   0.474    [0.201, 0.784]     0.980
------------------------------------------------------------------------------------------------
P(<0.80) = bootstrap probability the true impact ratio breaches the four-fifths rule.
Wide intervals indicate the audit is underpowered for that subgroup, not that it is safe.

Subgroup sizes in the 2,400-applicant test set:
intersection
Male / White                  820
Male / Asian                  524
Female / White                293
Female / Asian                215
Male / Hispanic/Latino        194
Male / Black                  134
Female / Hispanic/Latino       68
Female / Black                 60
Male / Two or more/Other       60
Female / Two or more/Other     32


**Two findings that only appear with intervals:**

1. Female / Hispanic-Latino shows an 87% bootstrap probability of a genuine four-fifths breach
   — a real finding, not noise, despite n=68.
2. Hispanic/Latino overall has a *passing* point estimate of 0.811 but a 46% probability the
   true ratio breaches. Reporting the point estimate alone would be misleading.

The governed model passes marginal audits for sex and race while still failing
intersectionally. This is precisely why LL144 mandates intersectional reporting.

## 4. Explainability

Two distinct jobs, often conflated:

- **Global** — what does the model rely on? A bias-detection tool.
- **Local** — why was *this* applicant scored this way? A legal artifact, feeding
  adverse-action-style notices required under Illinois AIVIA and the EU AI Act.

**Caveat that belongs in the code, not just the report:** SHAP explains the *model*, not the
*world*. A +0.30 attribution for "referral" means the model raised its score because the
applicant was referred. It does not mean referrals cause job success. Treating SHAP
attributions as causal is how an organization talks itself into believing a biased feature is
a legitimate one.

In [6]:
from explain import run as explain_run
shap_summary = explain_run()

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



GLOBAL SHAP IMPORTANCE — NAIVE MODEL
feature                           mean |SHAP|    share   proxy?
------------------------------------------------------------------------
Keyword match score                    0.6979    18.8%         
Employee referral                      0.5773    15.5%      YES
Relevant skills matched                0.4614    12.4%         
Employment gap (months)                0.3608     9.7%      YES
Education level                        0.3187     8.6%         
Attended 'top' school                  0.3056     8.2%      YES
Portfolio projects                     0.2921     7.9%         
Distance from office (km)              0.1576     4.2%      YES
Leadership indicators                  0.1488     4.0%         
Years of experience                    0.1012     2.7%         
------------------------------------------------------------------------
Share of total attribution carried by demographic proxy features: 37.7%



GLOBAL SHAP IMPORTANCE — GOVERNED MODEL
feature                           mean |SHAP|    share   proxy?
------------------------------------------------------------------------
Keyword match score                    0.7891    33.3%         
Relevant skills matched                0.4034    17.0%         
Education level                        0.2987    12.6%         
Portfolio projects                     0.2980    12.6%         
GPA                                    0.1504     6.3%         
Leadership indicators                  0.1154     4.9%         
Years of experience                    0.0870     3.7%         
Resume length (words)                  0.0751     3.2%         
Avg tenure (months)                    0.0712     3.0%         
Certifications                         0.0519     2.2%         
------------------------------------------------------------------------
Share of total attribution carried by demographic proxy features: 0.0%



GENERATED CANDIDATE-FACING EXPLANATION (score=0.297)
The four factors that most reduced this application's score:

  1. Education level: your value was 1  (impact -0.168)
  2. Leadership indicators: your value was 1  (impact -0.063)
  3. Prior roles: your value was 1  (impact -0.053)
  4. Certifications: your value was 1  (impact -0.014)

This text is machine-generated from SHAP attributions and is reviewed by a
human recruiter before release. It describes the model's reasoning, not a
judgment about the applicant's ability.


## 5. Guardrails and red teaming

Six layers. The load-bearing design decision: **the model cannot reject anyone.** It
produces a ranked shortlist and a routing recommendation; rejection remains a human act.
There is deliberately no code path returning `REJECT`.

Prompt injection is a live threat, not hypothetical: any pipeline passing resume text to an
LLM is one where *the applicant controls part of the prompt*.

In [7]:
from guardrails import run as guard_run
guard = guard_run()

                                  RED TEAM RESULTS                                  
id      scenario                                        expected              result
------------------------------------------------------------------------------------
RT-01   Direct prompt injection in resume body          prompt_injection      PASS
RT-02   Hidden white-on-white keyword block             hidden_text           PASS
RT-03   Keyword stuffing                                keyword_stuffing      PASS
RT-04   Zero-width character obfuscation                zero_width_characters PASS
RT-05   Role-reassignment injection                     prompt_injection      PASS
RT-06   Benign resume (must NOT trigger)                no flags              PASS
RT-07   Benign resume mentioning AI safety work (must   no flags              PASS
------------------------------------------------------------------------------------
7/7 red-team cases passed

                                  INPUT VALIDATION  

### A red-team finding that changed the design

Case **RT-07** originally failed. An early revision matched the bare phrase
`prompt injection`, which flagged legitimate ML-security engineers who simply described their
own work — a guardrail that discriminated against applicants in the AI safety field.

The fix: detection keys on *instructional framing directed at the system*, never on subject
matter. RT-07 is retained as a standing regression test. This is what red teaming is for —
the defect was in the defense, not the model.

## 6. Monitoring

Three failure modes, each needing its own detector. The one that matters most and is
monitored least is **fairness drift**: impact ratios degrade while AUC, PSI, and every
accuracy metric hold steady.

In [8]:
from monitor import run as monitor_run
history = monitor_run()

                            SIMULATED 6-MONTH PRODUCTION MONITORING                             
month      max PSI          drifted feat      AUC  sel.rate   IR(sex)   IR(race)      status
------------------------------------------------------------------------------------------------
0            0.000      years_experience    0.865     0.331     0.967      0.833          OK
1            0.016      years_experience    0.874     0.324     0.948      0.829          OK
2            0.065      years_experience    0.867     0.330     0.987      0.779    FAIRNESS


3            0.167      years_experience    0.873     0.323     0.934      0.941          OK
4            0.273      years_experience    0.876     0.309     0.946      0.930       DRIFT
5            0.441      years_experience    0.874     0.323     0.972      0.844       DRIFT
6            0.683      years_experience    0.868     0.312     0.966      0.810       DRIFT
------------------------------------------------------------------------------------------------
Over 6 simulated months:  AUC moved -0.003   race impact ratio moved -0.022
PSI alert threshold 0.25; fairness SLO 0.80.

                                         ALERT ROUTING                                          
trigger                                     sev   response
------------------------------------------------------------------------------------------------
PSI >= 0.10 on any feature                  P3    Data science reviews within 5 business days
PSI >= 0.25 on any feature                  P2    Retraining a

At month 2 the fairness SLO breaches (race impact ratio 0.779) while AUC sits at 0.867 and
max PSI is 0.065 — *below even the warning band*. A performance-only dashboard shows all
green during an active four-fifths breach.

By month 6 the inverse holds: PSI 0.683 (well past alert) with AUC unmoved at 0.868. Drift
alarms fire with no performance degradation. Both directions of the decoupling are real, and
neither is visible from a single dashboard.

## 7. Sustainability

In [9]:
from sustainability import run as sustain_run
sustain = sustain_run()

                                MEASURED RESOURCE FOOTPRINT                                 
model                   train CPU s   size KB   infer ms/1k   train Wh  train gCO2
--------------------------------------------------------------------------------------------
governed_logistic              0.02       2.1          0.65     0.0001      0.0000
governed_gbm                   0.40     201.9          3.17     0.0019      0.0007
naive_gbm                      0.36     240.9          3.24     0.0017      0.0006
--------------------------------------------------------------------------------------------
Assumptions: 15.0 W/core, PUE 1.12, grid 369.0 gCO2/kWh (US avg).

                     ANNUAL PIPELINE FOOTPRINT AT 250,000 RESUMES/YEAR                      
  Resume parsing / feature extraction         140.00 Wh   (120 ms/resume, assumed)
  API + serialization + logging                46.67 Wh   (40 ms/resume, assumed)
  Model scoring                                0.001 Wh   (measu

**A methodological correction worth stating.** An earlier revision counted only the
model's own arithmetic and produced a defensible-looking claim of an ~8-order-of-magnitude
advantage over an LLM. That was dishonest accounting. Priced across the whole serving
envelope — parsing, serialization, logging — the advantage is roughly 3 orders of magnitude.
Still decisive, and it has the advantage of being true.

The more useful finding: the model is **0.0004%** of pipeline energy. Optimizing it further
would be optimizing the wrong thing.

## 8. Conclusion

| Question | Answer |
|---|---|
| Does removing proxies fix disparate impact? | Largely — worst race impact ratio 0.63 → 0.83 |
| Does it fix it completely? | **No.** Female/Hispanic-Latino remains at 0.65 |
| Does fairness cost accuracy? | Against the biased label yes (−0.046 AUC); against true qualification **no**, it gains +0.046 |
| Is the system safe to deploy fully automated? | **No** — and the design forecloses it |

The honest conclusion is that this system is suitable for **ranking and routing under human
review**, not for automated rejection. A governance report that concluded otherwise would be
recommending something the evidence in this notebook does not support.